<a href="https://colab.research.google.com/github/jelenavasic009-png/dsb_capstone/blob/main/JelenaVasic_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import metrics

Korak 1: Učitavanje Excel fajla u Python

In [27]:
# Voditi racuna na "pd." da je ispred read_excel

df = pd.read_excel("/content/Baza podataka 5 godina.xlsx")
print("Podaci su uspješno učitani!")

Podaci su uspješno učitani!


In [19]:
# Da provjerimo kako podaci izgledaju

df.head(10)


,Name of the vessel,Vessel registration mark / Vessel ID,Length of the vessel,Purpose of use,Valid from,Valid until,Duration,Issued by,Price excluding tax €,Tax €,Total price €
Vignette serial number,,,,,,,,,,,
2020202103889,MELODY,744574,16.80,personal,2021-01-04,2021-01-05,"24, h",Bar,16.80,2,18.80
2020202102390,QUIEN SABE,DL9833AM,13.35,personal,2021-01-10,2021-01-10,"24, h",Tivat,40.05,2,42.05
2020202103890,GRETONY,745770,16.80,personal,2021-01-13,2021-01-20,"7, d",Bar,33.60,2,35.60
2020202100414,KAY AND DUKE,DL5964AL,15.24,personal,2021-01-13,2021-07-13,"6, M",Zelenika,381.00,2,383.00
2020202102391,ODYSSEY,40618,23.20,personal,2021-01-15,2021-01-16,"24, h",Tivat,23.20,2,25.20
2020202102393,ANNA MARIA I,34995-10 -D,17.20,personal,2021-01-16,2021-01-17,"24, h",Tivat,51.60,2,53.60
2020202102392,MURMUR,P001006,9.99,personal,2021-01-16,2021-01-23,"7, d",Tivat,19.98,2,21.98
2020202100416,AURORA,RUS 9201,11.94,personal,2021-01-17,2021-01-18,"24, h",Zelenika,11.94,2,13.94
2020202100415,VONGOLA,DL9998AM,11.90,personal,2021-01-17,2021-01-18,"24, h",Zelenika,11.90,2,13.90


In [36]:
# Ukupna veličina tabela

df.shape


(26764, 14)

In [21]:
# Tipovi podataka i prazna polja

df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 26764 entries, 2020202103889 to 15658
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Name of the vessel                    26758 non-null  object        
 1   Vessel registration mark / Vessel ID  26685 non-null  object        
 2   Length of the vessel                  26764 non-null  float64       
 3   Purpose of use                        26764 non-null  object        
 4   Valid from                            26764 non-null  datetime64[ns]
 5   Valid until                           26764 non-null  datetime64[ns]
 6   Duration                              26761 non-null  object        
 7   Issued by                             26764 non-null  object        
 8   Price excluding tax €                 26764 non-null  float64       
 9   Tax €                                 26764 non-null  int64      

In [38]:
#prvo provjeravam dje sve imam nule
df.isnull().sum()

,0
Vignette serial number,0
Name of the vessel,6
Vessel registration mark / Vessel ID,79
Length of the vessel,0
Purpose of use,0
Valid from,0
Valid until,0
Duration,3
Issued by,0
Price excluding tax €,0


In [22]:
#Statistika
df.describe()


,Length of the vessel,Valid from,Valid until,Price excluding tax €,Tax €,Total price €
count,26764.000000,26764,26764,26764.000000,26764.0,26764.000000
mean,15.376766,2023-08-18 17:04:28.480047872,2023-10-17 04:39:04.761620224,135.848941,2.0,137.848941
min,2.000000,2021-01-04 00:00:00,2021-01-05 00:00:00,1.660000,2.0,3.660000
25%,8.447500,2022-06-28 00:00:00,2022-08-15 00:00:00,22.600000,2.0,24.600000
50%,12.950000,2023-08-01 00:00:00,2023-09-28 00:00:00,66.570000,2.0,68.570000
75%,16.820000,2024-09-01 00:00:00,2024-11-18 00:00:00,176.780000,2.0,178.780000
max,162.500000,2026-01-20 00:00:00,2026-12-30 00:00:00,5700.000000,2.0,5702.000000
std,12.265739,NaN,NaN,193.603423,0.0,193.603423



Korak 2: Čišćenje i brisanje praznih redova

U Excel tabelama koje ljudi ručno popunjavaju često se na dnu nađu prazni redovi ili redovi tipa „Ukupno“ (kao što je red 70 u primjeru). Ti redovi bi zbunili naš model mašinskog učenja jer nemaju podatke o plovilima.

In [39]:
# Brišemo sve redove koji nemaju upisanu ispostavu (Izdata od) ili cijenu
df = df.dropna(subset=['Name of the vessel', 'Vessel registration mark / Vessel ID', 'Duration'])

#Objašnjenje: Funkcija dropna traži redove u kojima nedostaju podaci (NaN vrijednosti) u ključnim kolonama i trajno ih uklanja, čisteći bazu.

print("Uspješno obrisani prazni i zbirni redovi sa dna Excela!")


Uspješno obrisani prazni i zbirni redovi sa dna Excela!



Korak 3: Pretvaranje teksta u prave datume

U Excelu datumi često izgledaju kao običan tekst (npr. 2025-02-01T00:00:00). Python to vidi kao skup slova i brojeva, a ne kao vrijeme. Moramo mu reći da je to kalendarski datum da bi mogao da ga razumije.

In [40]:
# Pretvaramo kolonu 'Važi od' u zvanični vremenski format
df['Valid from'] = pd.to_datetime(df['Valid from'])

# Sada iz tog datuma izvlačimo posebne kolone za Godinu i Mjesec
df['Year'] = df['Valid from'].dt.year
df['Month'] = df['Valid from'].dt.month

#pd.to_datetime analizira tekst i pretvara ga u vremensku varijablu. Nakon toga, pomoću .dt.year i .dt.month uzimamo samo godinu i mjesec i stavljamo ih u potpuno nove kolone na kraju tabele.

print("Datumi su uspješno konvertovani! Napravljene su nove kolone 'Godina' i 'Mjesec'.")


Datumi su uspješno konvertovani! Napravljene su nove kolone 'Godina' i 'Mjesec'.


/tmp/ipykernel_685/1256364122.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Valid from'] = pd.to_datetime(df['Valid from'])
/tmp/ipykernel_685/1256364122.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Year'] = df['Valid from'].dt.year
/tmp/ipykernel_685/1256364122.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do


Korak 4: Agregacija (Grupisanje podataka za model)

Ovo je ključni korak  projekta. Trenutno tabela ima redove za pojedinačna plovila (jedan red = jedna prodata vinjeta). Međutim, naš model treba da predvidi ukupne mjesečne prihode i ukupnu količinu zaliha za cijelu godinu. Zato moramo da saberemo i prebrojimo podatke na nivou mjeseca za svaku ispostavu pojedinačno.

In [ ]:
# Grupisanje podataka po Ispostavi, Godini i Mjesecu
#pripremljeni_df = df.groupby(['Izdata od', 'Godina', 'Mjesec']).agg(
    #Ukupan_Prihod_Eur=('cijena_ukupno__', 'sum'),                  # Sabiramo sav novac u tom mjesecu
    #Potrebno_Univerzalnih_Naljepnica=('_id', 'count'),              # Brojimo koliko je ukupno naljepnica izdato
    #Prosjecna_Duzina_Plovila_Metara=('Dužina plovnog objekta', 'mean') # Računamo prosječnu dužinu brodova
#).reset_index()

# Prikazujemo kako izgleda nova tabela spremna za mašinsko učenje
#print("\n--- NOVA TABELA SPREMNA ZA MODELE MAŠINSKOG UČENJA ---")
#print(pripremljeni_df)


PLOT

In [41]:
# Grouping data by Office/Branch, Year, and Month (Grupisanje podataka po Ispostavi, Godini i Mjesecu)
prepared_df = df.groupby(['Valid from', 'Year', 'Month']).agg(
    Total_Revenue_Eur=('Total price €', 'sum'),                     # Summing up all revenue for that month (Sabiramo sav novac u tom mjesecu)
    Required_Universal_Vignettes=('Vignette serial number', 'count'),                    # Counting the total number of vignettes issued (Brojimo koliko je ukupno naljepnica izdato)
    Average_Vessel_Length_Meters=('Length of the vessel', 'mean')   # Calculating the average vessel length (Računamo prosječnu dužinu brodova)
).reset_index()

# Displaying what the new table looks like, ready for machine learning models (Prikazujemo kako izgleda nova tabela spremna za mašinsko učenje)
print("\n--- NEW DATASET READY FOR MACHINE LEARNING MODELS ---")
print(prepared_df)



--- NEW DATASET READY FOR MACHINE LEARNING MODELS ---
     Valid from  Year  Month  Total_Revenue_Eur  Required_Universal_Vignettes  \
0    2021-01-04  2021      1              18.80                             1   
1    2021-01-10  2021      1              42.05                             1   
2    2021-01-13  2021      1             418.60                             2   
3    2021-01-15  2021      1              25.20                             1   
4    2021-01-16  2021      1              75.58                             2   
...         ...   ...    ...                ...                           ...   
1662 2025-12-26  2025     12              84.11                             1   
1663 2025-12-27  2025     12             504.56                             4   
1664 2025-12-29  2025     12             983.22                             4   
1665 2025-12-30  2025     12             498.80                             5   
1666 2026-01-20  2026      1             105.20       


Korak 5: Čuvanje pripremljene baze u novi Excel fajl

Kada imam savršeno sređenu tabelu koja je spremna za treniranje algoritama mašinskog učenja, sačuvaću je u novi Excel fajl na računaru, kako ne bi morala svaki put ponovo da radim ove korake.

In [35]:
# Saving the processed DataFrame into a new Excel file (Čuvanje novog DataFrame-a u novi Excel fajl)
prepared_df.to_excel('vignette_inventory_and_revenue.xlsx', index=False)

print("Successfully created and saved the new Excel file: 'vignette_inventory_and_revenue.xlsx'!") #(zalihe_i_prihodi_ispostave)


Successfully created and saved the new Excel file: 'vignette_inventory_and_revenue.xlsx'!


• to_excel: This is the command I use to push all my newly grouped and cleaned data back into a fresh, readable Excel sheet.

• vignette_inventory_and_revenue.xlsx: This is just the name I am giving my new file so I remember it covers both my logistics goals (vignette count) and my financial goals (revenue).

• index=False: I need to include this so Python doesn't automatically add that annoying, messy column of row numbers (0, 1, 2, 3...) into my final Excel file. This keeps my spreadsheet clean and ready for my project presentation.


• to_excel: Ovo je komanda koju koristim da sve svoje novo-grupisane i očišćene podatke prebacim nazad u novi, pregledni Excel list.

• vignette_inventory_and_revenue.xlsx: Ovo je samo ime koje dajem svom novom fajlu kako bih zapamtila da on pokriva i moje logističke ciljeve (broj vinjeta) i moje finansijske ciljeve (prihod).

• index=False: Ovo moram da uključim kako Python ne bi automatski dodao onu dosadnu, neurednu kolonu sa brojevima redova (0, 1, 2, 3...) u moj konačni Excel fajl. Tako mi tabela ostaje čista i spremna za prezentaciju projekta.
